In [184]:
from bell import *
import scipy as sc

from cirq_sic.wh import wh_povm
from cirq_sic.sics import load_sic_fiducial
from cirq_sic.utils import rand_ket

In [185]:
aligned, details = test_ordering_alignment(w)
print(aligned, details)

True {'aligned': True, 'tested_indices': [0, 15, 31, 47, 63], 'num_lambdas': 64, 'num_probabilities': 36, 'mismatches': []}


In [195]:
d = 2
n_parties = 2
w = [[d, d**2], [d, d**2]]
deterministic_behaviors = construct_deterministic_behaviors(w)

basis_povm = np.array([np.diag(np.eye(d)[i]) for i in range(d)])
U = sc.stats.unitary_group.rvs(d)
basis_povm2 = [U @ _ @ U.conj().T for _ in basis_povm]
sic_povm = wh_povm(load_sic_fiducial(d))

#E = [[basis_povm, sic_povm], [transpose_povm(basis_povm), transpose_povm(sic_povm)]]
#E = [[basis_povm, sic_povm], [basis_povm, sic_povm]]
#E = [[sic_povm, transpose_povm(sic_povm)], [sic_povm, transpose_povm(sic_povm)]]
E = [[basis_povm, sic_povm], [basis_povm2, transpose_povm(sic_povm)]]

#ket = sum([np.kron(np.eye(d)[i], np.eye(d)[i]) for i in range(d)])/np.sqrt(d)
#ket = np.kron(sc.stats.unitary_group.rvs(d), np.eye(d)) @ ket
ket = rand_ket(d**2)
rho = np.outer(ket, ket.conj())

p = quantum_behavior_from_povms(E, rho)
bell_functional, classical_bound, problem = construct_bell_inequality(w, p, dichotomous=False, return_problem=True)

print("LP objective:", problem.value)
print("max(D @ s - S):", np.max(deterministic_behaviors @ bell_functional - classical_bound))
print("p·s, S:", float(p @ bell_functional), float(classical_bound))

LP objective: -2.5826008858145316e-11
max(D @ s - S): 1.4108118761150173e-11
p·s, S: 2.462068670569615e-06 2.462094496578473e-06


In [196]:
bell_functional @ p >= classical_bound

np.False_

In [103]:
r = abs(np.random.randn(deterministic_behaviors.shape[0]))
r = r/np.sum(r)
classical_p = r @ deterministic_behaviors
classical_p @ bell_functional

np.float64(-0.027925472500796904)

In [104]:
reference_measurements = np.array([wh_povm(load_sic_fiducial(d)) for i in range(n_parties)])
reference_states = np.array([d*reference_measurements[i] for i in range(n_parties)])

T, T_meta = construct_T(rho, E, reference_measurements, reference_states, return_metadata=True)
phi = T_meta["phi"]
assert np.allclose(p, T @ phi)
quantumness_bound(bell_functional, classical_bound, T, phi)

np.float64(-6.973766424555233e-10)

In [93]:
T_singular_values = np.linalg.svd(T, compute_uv=False)
Delta = bell_functional @ p - classical_bound
bound = Delta/(np.max(T_singular_values)*np.linalg.norm(bell_functional)); bound

np.float64(-4.775183501884811e-11)

In [94]:
bell_functional @ p -classical_bound

np.float64(-3.780430829491976e-12)

In [95]:
T_singular_values, np.linalg.norm(bell_functional)

(array([0.8660254038, 0.4409585518, 0.4409585518, 0.2245251047,
        0.1666666667, 0.1666666667, 0.1666666667, 0.1666666667,
        0.0848625129, 0.0848625129, 0.0848625129, 0.0848625129,
        0.032075015 , 0.032075015 , 0.032075015 , 0.032075015 ,
        0.          , 0.          , 0.          , 0.          ,
        0.          , 0.          , 0.          , 0.          ,
        0.          , 0.          , 0.          , 0.          ,
        0.          , 0.          , 0.          , 0.          ,
        0.          , 0.          , 0.          , 0.          ]),
 np.float64(0.09141565999289654))

In [96]:
(np.max(T_singular_values)*np.linalg.norm(bell_functional))

np.float64(0.07916828385756912)